In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/bank_transaction_data_unique.csv")
df.head()


,transaction_id,customer_id,transaction_date,merchant_category,channel,device_id,transaction_amount,is_repeat_customer,days_since_last_txn,is_fraud,age,gender,region,account_type,income_band,risk_score,risk_band
0,TXN00001,CUST0613,2025-08-04,Electronics,Mobile App,fefbc06e-0f41-4184-a29a-ac0eced8673c,430.74,1,3,0,35,Male,West,Savings,>200K,516.89,Medium
1,TXN00019,CUST0613,2025-02-16,Groceries,ATM,1b544957-640d-4388-91b2-f0c216f455f4,573.90,1,3,0,35,Male,West,Savings,>200K,688.68,Medium
2,TXN01948,CUST0613,2025-06-04,Clothing,Online,f4cd270c-bb95-4487-9379-59d95d538aae,92.87,1,6,0,35,Male,West,Savings,>200K,111.44,Low
3,TXN02290,CUST0613,2025-04-27,Travel,POS,7e9175dc-8939-45bd-ba69-99b5166dd13a,884.95,1,4,0,35,Male,West,Savings,>200K,1061.94,Medium
4,TXN02523,CUST0613,2025-07-04,Clothing,POS,3d119252-7083-4f35-b6df-ec7adf0bc533,808.67,1,3,0,35,Male,West,Savings,>200K,970.40,Medium


In [ ]:
# dataset size
df.shape

df.info()

df.describe()

df.isnull().sum()

df.drop_duplicates(inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       5000 non-null   object 
 1   customer_id          5000 non-null   object 
 2   transaction_date     5000 non-null   object 
 3   merchant_category    5000 non-null   object 
 4   channel              5000 non-null   object 
 5   device_id            5000 non-null   object 
 6   transaction_amount   5000 non-null   float64
 7   is_repeat_customer   5000 non-null   int64  
 8   days_since_last_txn  5000 non-null   int64  
 9   is_fraud             5000 non-null   int64  
 10  age                  5000 non-null   int64  
 11  gender               5000 non-null   object 
 12  region               5000 non-null   object 
 13  account_type         5000 non-null   object 
 14  income_band          5000 non-null   object 
 15  risk_score           5000 non-null   f

In [3]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

df['transaction_date'].isna().sum()


np.int64(0)

In [4]:
for col in ['merchant_category', 'channel', 'device_id', 'region', 'account_type', 'income_band', 'gender']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# if amount has null, set to 0
if 'transaction_amount' in df.columns:
    df['transaction_amount'] = df['transaction_amount'].fillna(0)


In [9]:
df.sort_values(['customer_id', 'transaction_date'], inplace=True)


In [12]:
df['days_since_last_txn'] = (
    df.groupby('customer_id')['transaction_date']
      .diff()
      .dt.days
      .fillna(0)
)
df['high_frequency_flag'] = (df['days_since_last_txn'] <= 2).astype(int)
# find a high cap at 99th percentile (protect against crazy values)
cap = df['transaction_amount'].quantile(0.99)
df['transaction_amount_capped'] = df['transaction_amount'].clip(upper=cap)




In [13]:
# if your data already has is_fraud column, keep it, else create (0 for all)
if 'is_fraud' not in df.columns:
    df['is_fraud'] = 0

df['risk_score'] = (
    df['transaction_amount_capped'] *
    (1 + 0.5*df['is_fraud'] + 0.2*df['high_frequency_flag'])
).round(2)

def risk_band(x):
    if x >= 1500: return 'High'
    if x >= 500:  return 'Medium'
    return 'Low'

df['risk_band'] = df['risk_score'].apply(risk_band)


In [14]:
# top categories
df['merchant_category'].value_counts().head(10)

# channel usage
df['channel'].value_counts()

# regions distribution
df['region'].value_counts()

# risk bands share
df['risk_band'].value_counts(normalize=True).round(3)

# monthly volume (useful later in Power BI)
monthly = df.set_index('transaction_date').resample('M')['transaction_id'].count()
monthly.head()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_21448\2630776899.py:14: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.set_index('transaction_date').resample('M')['transaction_id'].count()


transaction_date
2025-02-28    609
2025-03-31    833
2025-04-30    843
2025-05-31    819
2025-06-30    753
Freq: ME, Name: transaction_id, dtype: int64

In [15]:
df.to_csv("../data/cleaned_bank_transactions.csv", index=False)


In [16]:
# region-month summary (super handy in Excel/Power BI)
summary = (
    df.assign(month=df['transaction_date'].dt.to_period('M').astype(str))
      .groupby(['region','month','risk_band'], as_index=False)
      .agg(
          transactions=('transaction_id','count'),
          total_amount=('transaction_amount_capped','sum')
      )
)
summary.to_csv("../data/summary_region_month.csv", index=False)
summary.head()


,region,month,risk_band,transactions,total_amount
0,East,2025-02,High,2,2774.890
1,East,2025-02,Low,98,16445.650
2,East,2025-02,Medium,30,20552.060
3,East,2025-03,High,1,1387.445
4,East,2025-03,Low,154,27778.650
